# Duckboat Demo

*Ugly to some, but gets the job done.*

Chain SQL snippets into composable data pipelines with DuckDB.

In [ ]:
!pip install -q duckboat

In [ ]:
import duckboat as uck

## Chaining SQL snippets

Each step takes the previous result as input. `from _` is prepended automatically.

In [ ]:
penguins = uck.examples.penguins()
penguins

In [ ]:
penguins.do(
    "where sex = 'female'",
    'where body_mass_g is not null',
    'select species, island, avg(body_mass_g) as avg_mass group by 1, 2',
    'select * replace (round(avg_mass, 1) as avg_mass)',
    'order by avg_mass',
)

## Materializing results

Extract scalars, lists, or DataFrames from the chain.

In [ ]:
penguins.do('select count(*)', int)

In [ ]:
penguins.do('select distinct species', list)

In [ ]:
penguins.do('where species = \'Adelie\'', 'limit 3', 'pandas')

## Joins

Pass a dict to register named tables.

In [ ]:
store = uck.examples.store()

uck.do(
    store,
    'select name, sum(amount) as total from orders join customers on customer_id = customers.id group by 1',
    'order by total desc',
    'limit 5',
)

## Mid-chain joins

In [ ]:
store['orders'].do(
    'where amount > 10',
    {'customers': store['customers']},
    'join customers on customer_id = customers.id',
    'select name, sum(amount) as total group by 1',
    'order by total desc',
)

## Self-joins

Use `uck.rename()` to give the current table a name, then write full SQL.

In [ ]:
penguins.do(
    'select distinct species',
    uck.rename('sp'),
    'select a.species as sp1, b.species as sp2 from sp a cross join sp b where a.species < b.species',
)

## Reusable fragments

Store pipeline steps as lists.

In [ ]:
clean = [
    'where body_mass_g is not null',
    'where sex is not null',
]

summarize = [
    'select species, sex, avg(body_mass_g) as avg_mass group by 1, 2',
    'select * replace (round(avg_mass, 1) as avg_mass)',
    'order by species, sex',
]

penguins.do(clean, summarize)

## Interop with pandas

Pop into pandas for operations that are easier there, then come back.

In [ ]:
df = penguins.do('select species, body_mass_g', 'pandas')
df.columns = df.columns.str.upper()

uck.do(df, 'select SPECIES, avg(BODY_MASS_G) as avg group by 1')